# Feature Exploration: Comparing rainy_days_arr Transformations

This notebook explores the idea of adding weather features back to ML models, starting with `rainy_days_arr`, which quantifies how rainy a certain month is. It also compares different transformations `rainy_days_arr` to see which works best.:
1. `rainy_days_arr` - plain (no transformation)
2. `rainy_days_arr_exp` - exponentially scaled: the idea of using exponential scaling is to amplify the effects of high number of rainy days

The model developed in 4b (using features: airline + cyclical month + lag1 + sectors_scheduled) is used as the baseline model. Similar to notebook 4b, the following models will be tested in this one:
- Regression: Ridge, Random Forest
- Classification: XGBoost, Random Forest, Logistic Regression

**Note:** Tree-based models (RF, XGBoost) are invariant to monotonic transformations, so differences should only appear in linear models (Ridge, Logistic).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Change plot font
plt.rcParams['font.family'] = 'serif'

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed (pip install xgboost)")

%matplotlib inline

In [ ]:
import sys; print(sys.executable)

## 1. Data Preparation

Same as the previous notebook.

In [ ]:
# Load data with weather features
df = pd.read_csv('../data/processed/ml_training_data_syd_mel_with_holidays.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = pd.to_datetime(df['month']).dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']

# Sort for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

Except here, the rainy days features (plain and exponential) are also added.

In [ ]:
# Create lagged feature (previous month's delay rate for same airline-route)
df['delay_rate_lag1'] = df.groupby('airline_route')['delay_rate'].shift(1)

# Create cyclical month encoding
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df['airline'], prefix='airline')
df = pd.concat([df, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

# Create exponential transformation of rainy_days_arr
df['rainy_days_arr_exp'] = np.exp(df['rainy_days_arr'] / df['rainy_days_arr'].max())

# Observe the range difference
print(f"rainy_days_arr range: {df['rainy_days_arr'].min():.1f} - {df['rainy_days_arr'].max():.1f}")
print(f"rainy_days_arr_exp range: {df['rainy_days_arr_exp'].min():.2f} - {df['rainy_days_arr_exp'].max():.2f}")

# Drop rows with missing lag values
df_clean = df.dropna(subset=['delay_rate_lag1']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

### Train/Validation/Test Split

As discussed in the previous notebook, split the training data stratified across pre and post-COVID years:
* Train: 2010-2017 and 2024 for actual model training
* Validation: 2018 and 2023 for cross-validation (tuning hyperparameters)
* Test: 2019 and 2025 for final evaluation

In [ ]:
# Time-based split (excluding 2020-2022 COVID period)
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")
print("\nNote: 2020-2022 excluded (COVID period)")

In [ ]:
# Define feature sets for comparison
base_features = airline_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

feature_variants = {
    'plain': base_features + ['rainy_days_arr'],
    'exp': base_features + ['rainy_days_arr_exp'],
}

for name, features in feature_variants.items():
    print(f"{name}: {len(features)} features")

In [ ]:
# Prepare target variables (same for all variants)
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")

## 2. Train Models for Each Feature Variant

In [ ]:
# Train all models for each feature variant
all_results = []

for variant_name, features in feature_variants.items():
    print(f"\n{'='*60}")
    print(f"Training with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data for this variant
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Ridge Regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train_reg)
    ridge_test_pred = ridge.predict(X_test_scaled)
    
    ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
    print(f"  Ridge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}")
    
    all_results.append({
        'Variant': variant_name,
        'Model': 'Ridge',
        'Type': 'Regression',
        'Test_R2': ridge_r2,
        'Test_RMSE': ridge_rmse
    })
    
    # Random Forest Regression
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train, y_train_reg)
    rf_test_pred = rf_reg.predict(X_test)
    
    rf_r2 = r2_score(y_test_reg, rf_test_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
    print(f"  RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}")
    
    all_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Type': 'Regression',
        'Test_R2': rf_r2,
        'Test_RMSE': rf_rmse
    })
    
    # Logistic Regression
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_test_pred = logreg.predict(X_test_scaled)
    logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
    
    logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
    logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
    print(f"  Logistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}")
    
    all_results.append({
        'Variant': variant_name,
        'Model': 'Logistic',
        'Type': 'Classification',
        'Test_F1': logreg_f1,
        'Test_AUC': logreg_auc
    })
    
    # Random Forest Classification
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_test_pred = rf_clf.predict(X_test)
    rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
    
    rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
    rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
    print(f"  RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}")
    
    all_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Type': 'Classification',
        'Test_F1': rf_clf_f1,
        'Test_AUC': rf_clf_auc
    })
    
    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            min_child_weight=5, random_state=42, n_jobs=-1
        )
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_test_pred = xgb_clf.predict(X_test)
        xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
        
        xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
        xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
        print(f"  XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}")
        
        all_results.append({
            'Variant': variant_name,
            'Model': 'XGBoost',
            'Type': 'Classification',
            'Test_F1': xgb_f1,
            'Test_AUC': xgb_auc
        })

results_df = pd.DataFrame(all_results)
print("\nAll models trained.")

## 3. Side-by-Side Comparison

In [ ]:
# Pivot results for side-by-side comparison
print("=" * 80)
print("REGRESSION: plain vs exp (rainy_days_arr)")
print("=" * 80)

reg_df = results_df[results_df['Type'] == 'Regression'].copy()
reg_pivot = reg_df.pivot(index='Model', columns='Variant', values=['Test_R2', 'Test_RMSE'])

# Flatten column names
reg_pivot.columns = [f'{col[1]}_{col[0]}' for col in reg_pivot.columns]
reg_pivot = reg_pivot.reset_index()

# Calculate differences
reg_pivot['R2_diff'] = reg_pivot['exp_Test_R2'] - reg_pivot['plain_Test_R2']
reg_pivot['RMSE_diff'] = reg_pivot['exp_Test_RMSE'] - reg_pivot['plain_Test_RMSE']

print(f"\n{'Model':<15} {'plain R²':>10} {'exp R²':>10} {'R² diff':>10} {'plain RMSE':>12} {'exp RMSE':>10} {'RMSE diff':>10}")
print("-" * 80)
for _, row in reg_pivot.iterrows():
    r2_sign = '+' if row['R2_diff'] > 0 else ''
    rmse_sign = '+' if row['RMSE_diff'] > 0 else ''
    print(f"{row['Model']:<15} {row['plain_Test_R2']:>10.4f} {row['exp_Test_R2']:>10.4f} {r2_sign}{row['R2_diff']:>9.4f} {row['plain_Test_RMSE']:>12.4f} {row['exp_Test_RMSE']:>10.4f} {rmse_sign}{row['RMSE_diff']:>9.4f}")

In [ ]:
# Classification comparison
print("=" * 80)
print("CLASSIFICATION: plain vs exp (rainy_days_arr)")
print("=" * 80)

clf_df = results_df[results_df['Type'] == 'Classification'].copy()
clf_pivot = clf_df.pivot(index='Model', columns='Variant', values=['Test_F1', 'Test_AUC'])

# Flatten column names
clf_pivot.columns = [f'{col[1]}_{col[0]}' for col in clf_pivot.columns]
clf_pivot = clf_pivot.reset_index()

# Calculate differences
clf_pivot['F1_diff'] = clf_pivot['exp_Test_F1'] - clf_pivot['plain_Test_F1']
clf_pivot['AUC_diff'] = clf_pivot['exp_Test_AUC'] - clf_pivot['plain_Test_AUC']

print(f"\n{'Model':<15} {'plain F1':>10} {'exp F1':>10} {'F1 diff':>10} {'plain AUC':>12} {'exp AUC':>10} {'AUC diff':>10}")
print("-" * 80)
for _, row in clf_pivot.iterrows():
    f1_sign = '+' if row['F1_diff'] > 0 else ''
    auc_sign = '+' if row['AUC_diff'] > 0 else ''
    print(f"{row['Model']:<15} {row['plain_Test_F1']:>10.4f} {row['exp_Test_F1']:>10.4f} {f1_sign}{row['F1_diff']:>9.4f} {row['plain_Test_AUC']:>12.4f} {row['exp_Test_AUC']:>10.4f} {auc_sign}{row['AUC_diff']:>9.4f}")

In [ ]:
# Visualization: Side-by-side bar charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Regression R²
ax = axes[0, 0]
x = np.arange(len(reg_pivot))
width = 0.35
ax.bar(x - width/2, reg_pivot['plain_Test_R2'], width, label='plain', color='steelblue', alpha=0.8)
ax.bar(x + width/2, reg_pivot['exp_Test_R2'], width, label='exp', color='#e74c3c', alpha=0.8)
ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(reg_pivot['Model'])
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

# Regression RMSE
ax = axes[0, 1]
ax.bar(x - width/2, reg_pivot['plain_Test_RMSE'], width, label='plain', color='steelblue', alpha=0.8)
ax.bar(x + width/2, reg_pivot['exp_Test_RMSE'], width, label='exp', color='#e74c3c', alpha=0.8)
ax.set_ylabel(r'Test RMSE')
ax.set_title(r'Regression: RMSE Comparison (lower is better)')
ax.set_xticks(x)
ax.set_xticklabels(reg_pivot['Model'])
ax.legend()
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

# Classification F1
ax = axes[1, 0]
x = np.arange(len(clf_pivot))
ax.bar(x - width/2, clf_pivot['plain_Test_F1'], width, label='plain', color='steelblue', alpha=0.8)
ax.bar(x + width/2, clf_pivot['exp_Test_F1'], width, label='exp', color='#e74c3c', alpha=0.8)
ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(clf_pivot['Model'])
ax.legend()
ax.set_ylim(0, 1)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

# Classification AUC
ax = axes[1, 1]
ax.bar(x - width/2, clf_pivot['plain_Test_AUC'], width, label='plain', color='steelblue', alpha=0.8)
ax.bar(x + width/2, clf_pivot['exp_Test_AUC'], width, label='exp', color='#e74c3c', alpha=0.8)
ax.set_ylabel(r'Test AUC')
ax.set_title(r'Classification: AUC Comparison')
ax.set_xticks(x)
ax.set_xticklabels(clf_pivot['Model'])
ax.legend()
ax.set_ylim(0, 1)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.show()

## 4. Summary and Observations

In [ ]:
# Summary
print("=" * 80)
print("SUMMARY: plain vs exp (rainy_days_arr)")
print("=" * 80)

# Linear models only (where transformation matters)
print("\nLINEAR MODELS (where transformation matters):")
print("-" * 50)

# Ridge
ridge_plain = reg_pivot[reg_pivot['Model'] == 'Ridge']['plain_Test_R2'].values[0]
ridge_exp = reg_pivot[reg_pivot['Model'] == 'Ridge']['exp_Test_R2'].values[0]
ridge_winner = 'exp' if ridge_exp > ridge_plain else 'plain'
print(f"  Ridge Regression: {ridge_winner} wins (R² diff: {ridge_exp - ridge_plain:+.4f})")

# Logistic
log_plain = clf_pivot[clf_pivot['Model'] == 'Logistic']['plain_Test_F1'].values[0]
log_exp = clf_pivot[clf_pivot['Model'] == 'Logistic']['exp_Test_F1'].values[0]
log_winner = 'exp' if log_exp > log_plain else 'plain'
print(f"  Logistic Regression: {log_winner} wins (F1 diff: {log_exp - log_plain:+.4f})")

# Tree models (should be similar)
print("\nTREE MODELS (should be similar due to monotonic invariance):")
print("-" * 50)

rf_reg_plain = reg_pivot[reg_pivot['Model'] == 'Random Forest']['plain_Test_R2'].values[0]
rf_reg_exp = reg_pivot[reg_pivot['Model'] == 'Random Forest']['exp_Test_R2'].values[0]
print(f"  RF Regression: diff = {rf_reg_exp - rf_reg_plain:+.4f}")

rf_clf_plain = clf_pivot[clf_pivot['Model'] == 'Random Forest']['plain_Test_F1'].values[0]
rf_clf_exp = clf_pivot[clf_pivot['Model'] == 'Random Forest']['exp_Test_F1'].values[0]
print(f"  RF Classification: diff = {rf_clf_exp - rf_clf_plain:+.4f}")

if 'XGBoost' in clf_pivot['Model'].values:
    xgb_plain = clf_pivot[clf_pivot['Model'] == 'XGBoost']['plain_Test_F1'].values[0]
    xgb_exp = clf_pivot[clf_pivot['Model'] == 'XGBoost']['exp_Test_F1'].values[0]
    print(f"  XGBoost Classification: diff = {xgb_exp - xgb_plain:+.4f}")


### Comparison with Baseline (4b)

Baseline results from 4b (without any rainy_days feature):

| Model | Type | Test R² | Test RMSE | Test F1 | Test AUC |
|-------|------|---------|-----------|---------|----------|
| Ridge Regression | Regression | 0.3932 | 0.0770 | - | - |
| Random Forest | Regression | 0.4163 | 0.0755 | - | - |
| Logistic Regression | Classification | - | - | 0.7529 | 0.8750 |
| Random Forest | Classification | - | - | 0.7807 | 0.8455 |
| XGBoost | Classification | - | - | 0.7876 | 0.8206 |

In [ ]:
# Compare best variant with baseline
baseline_reg = {
    'Ridge': {'Test_R2': 0.3932, 'Test_RMSE': 0.0770},
    'Random Forest': {'Test_R2': 0.4163, 'Test_RMSE': 0.0755}
}

baseline_clf = {
    'Logistic': {'Test_F1': 0.7529, 'Test_AUC': 0.8750},
    'Random Forest': {'Test_F1': 0.7807, 'Test_AUC': 0.8455},
    'XGBoost': {'Test_F1': 0.7876, 'Test_AUC': 0.8206}
}

print("=" * 80)
print("COMPARISON WITH BASELINE (4b - no rainy_days feature)")
print("=" * 80)

print("\nREGRESSION (best of plain/exp vs baseline):")
print(f"{'Model':<15} {'Baseline R²':>12} {'plain R²':>10} {'exp R²':>10} {'Best':>8} {'vs Base':>10}")
print("-" * 70)
for _, row in reg_pivot.iterrows():
    model = row['Model']
    if model in baseline_reg:
        base = baseline_reg[model]['Test_R2']
        plain = row['plain_Test_R2']
        exp = row['exp_Test_R2']
        best = max(plain, exp)
        best_name = 'exp' if exp > plain else 'plain'
        diff = best - base
        sign = '+' if diff > 0 else ''
        print(f"{model:<15} {base:>12.4f} {plain:>10.4f} {exp:>10.4f} {best_name:>8} {sign}{diff:>9.4f}")

print("\nCLASSIFICATION (best of plain/exp vs baseline):")
print(f"{'Model':<15} {'Baseline F1':>12} {'plain F1':>10} {'exp F1':>10} {'Best':>8} {'vs Base':>10}")
print("-" * 70)
for _, row in clf_pivot.iterrows():
    model = row['Model']
    if model in baseline_clf:
        base = baseline_clf[model]['Test_F1']
        plain = row['plain_Test_F1']
        exp = row['exp_Test_F1']
        best = max(plain, exp)
        best_name = 'exp' if exp > plain else 'plain'
        diff = best - base
        sign = '+' if diff > 0 else ''
        print(f"{model:<15} {base:>12.4f} {plain:>10.4f} {exp:>10.4f} {best_name:>8} {sign}{diff:>9.4f}")

### Observations

**Adding rainy_days_arr (vs baseline from 4b):**
- Regression models are improved: Both Ridge (+0.0195 R²) and RF (+0.0219 R²) benefit from adding rainy days feature. It brings the test R² value of Ridge model to 0.4127.
- Classification models have mixed results: Logistic improved slightly (+0.0061 F1), but RF and XGBoost decreased (-0.026 and -0.031 F1)

**Plain vs Exponential Transformation:**
- Ridge Regression is improved: Exponential transformation improves R² by +0.0071 (0.4056 → 0.4127)
- Logistic Regression is mixed: Plain version has higher F1 (+0.0151), but exp has slightly better AUC (+0.0035)
- Tree-based models (RF, XGBoost): No difference as expected, due to monotonic invariance


## 5. Next step

Based on the observations, the ML models moving forward will also include:
- Use `rainy_days_arr_exp` for regression models
- Use `rainy_days_arr` (plain) for classification models

Next, try to improve the lag feature, by adding the latest rate-of-change (gradient) of the delay rate. This potentially gives an indication of the delay rate's "momentum" (slowing down/ramping up/etc.).

See [4d_feature_exploration_momentum.ipynb](//notebooks/4d_feature_exploration_momentum.ipynb)
